Notebook 4 - relevance score

# Notebook 4 - Relevance Score

- **Approach A - Weighted linear score**: combine features using deseasonalized
  correlation strengths as weights. Simple, interpretable, directly grounded in findings.
- **Approach B - Sentiment-first score**: weight consistent sentiment more heavily
  than engagement. Tests whether why people are talking matters more than how much.
- **Approach C - Engagement momentum score**: focus on change in engagement
  rather than level. A destination going from low to high buzz is more actionable
  than one that has always been high.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:

OUTPUT_PATH = Path(r"path/to/your/data")

panel = pl.read_parquet(OUTPUT_PATH / "panel_features.parquet")
panel_pd = panel.to_pandas().sort_values(["lhg_country", "date"]).reset_index(drop=True)

print(f"Panel: {panel_pd.shape}")
print(f"Countries: {panel_pd['lhg_country'].nunique()}")
print(f"Date range: {panel_pd['date'].min()} → {panel_pd['date'].max()}")

## Normalise features per country

In [ ]:
SCORE_FEATURES = [
    "eng_score_roll7sum",
    "eng_score_roll28sum",
    "consistent_sentiment_score",
    "total_engagement_score",
    "engagement_momentum",
    "sentiment_momentum",
]

scaler = MinMaxScaler()
countries = panel_pd["lhg_country"].unique()

norm_frames = []
for country in countries:
    mask = panel_pd["lhg_country"] == country
    df_c = panel_pd[mask].copy()
    for feat in SCORE_FEATURES:
        if feat in df_c.columns:
            vals = df_c[feat].values.reshape(-1, 1)
            if vals.std() > 0:
                df_c[f"{feat}_norm"] = scaler.fit_transform(vals).flatten()
            else:
                df_c[f"{feat}_norm"] = 0.0
    norm_frames.append(df_c)

panel_scored = pd.concat(norm_frames).sort_values(["lhg_country", "date"]).reset_index(drop=True)
print("Normalised features:")
print([c for c in panel_scored.columns if c.endswith("_norm")])

## Scoring approaches (3 methods)

In [ ]:
# Approach A: Weighted linear
# Weights proportional to deseasonalized Spearman r at lag7
W_A = {
    "eng_score_roll7sum_norm":         0.201,
    "eng_score_roll28sum_norm":        0.200,
    "consistent_sentiment_score_norm": 0.147,
    "total_engagement_score_norm":     0.107,
}
total_w_a = sum(W_A.values())
W_A = {k: v / total_w_a for k, v in W_A.items()}

panel_scored["score_A"] = sum(
    panel_scored[feat] * w for feat, w in W_A.items()
)

print("Approach A — Weighted linear (weights from deseasonalized r):")
for k, v in W_A.items():
    print(f"  {k:<45} {v:.3f}")


# Approach B: Sentiment-first
# Tests: does weighting sustained sentiment more heavily improve predictive power?
W_B = {
    "eng_score_roll7sum_norm":         0.150,
    "eng_score_roll28sum_norm":        0.150,
    "consistent_sentiment_score_norm": 0.400,
    "total_engagement_score_norm":     0.100,
    "sentiment_momentum_norm":         0.200,
}
panel_scored["score_B"] = sum(
    panel_scored[feat] * w
    for feat, w in W_B.items()
    if feat in panel_scored.columns
)

print("\nApproach B — Sentiment-first:")
for k, v in W_B.items():
    print(f"  {k:<45} {v:.3f}")


# Approach C: Engagement momentum
# Tests: is a destination 'heating up' more predictive than absolute buzz level?
panel_scored["eng_momentum_pos_norm"] = panel_scored["engagement_momentum_norm"].clip(lower=0)

W_C = {
    "eng_momentum_pos_norm":           0.500,
    "eng_score_roll7sum_norm":         0.300,
    "consistent_sentiment_score_norm": 0.200,
}
panel_scored["score_C"] = sum(
    panel_scored[feat] * w
    for feat, w in W_C.items()
    if feat in panel_scored.columns
)

print("\nApproach C — Engagement momentum:")
for k, v in W_C.items():
    print(f"  {k:<45} {v:.3f}")

print("\nAll three scores computed.")

## Evaluate approaches

Ground truth: does a booking spike (pax_spike_positive) occur
within 14 days of the score exceeding a threshold?
We evaluate precision, recall, F1 and AUC.

In [ ]:
EVAL_WINDOW = 14
APPROACHES  = ["score_A", "score_B", "score_C"]

def build_forward_spike_label(df, spike_col, window):
    label = pd.Series(0, index=df.index)
    for country in df["lhg_country"].unique():
        mask  = df["lhg_country"] == country
        spike = df.loc[mask, spike_col].values
        n     = len(spike)
        fwd   = np.zeros(n, dtype=int)
        for i in range(n):
            if spike[i+1:i+window+1].sum() > 0:
                fwd[i] = 1
        label.loc[mask] = fwd
    return label

print(f"Building forward spike labels (window={EVAL_WINDOW}d)")
panel_scored["future_spike"] = build_forward_spike_label(
    panel_scored, "pax_spike_positive", EVAL_WINDOW
)
print(f"Rows with future spike: {panel_scored['future_spike'].sum():,}")
print(f"Base rate: {panel_scored['future_spike'].mean():.1%}")

In [ ]:
THRESHOLDS   = np.arange(0.1, 0.91, 0.05)
eval_results = []
y_true       = panel_scored["future_spike"].values

for approach in APPROACHES:
    scores = panel_scored[approach].fillna(0).values
    try:
        auc = roc_auc_score(y_true, scores)
    except Exception:
        auc = np.nan

    for thresh in THRESHOLDS:
        y_pred = (scores >= thresh).astype(int)
        if y_pred.sum() == 0:
            continue
        eval_results.append({
            "approach":  approach,
            "threshold": round(thresh, 2),
            "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
            "auc":       round(auc, 4),
            "n_flagged": int(y_pred.sum()),
        })

eval_df = pd.DataFrame(eval_results)

print("Best threshold per approach (max F1)")
best = eval_df.loc[eval_df.groupby("approach")["f1"].idxmax()]
print(best[["approach", "threshold", "precision", "recall", "f1", "auc"]].to_string(index=False))

print("\nAUC comparison")
print(eval_df.groupby("approach")["auc"].first().sort_values(ascending=False))

In [ ]:
# Precision-Recall curves
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors    = ["steelblue", "darkorange", "green"]
base_rate = panel_scored["future_spike"].mean()

for ax, approach, color in zip(axes, APPROACHES, colors):
    df_a     = eval_df[eval_df["approach"] == approach].sort_values("threshold")
    auc_val  = df_a["auc"].iloc[0]
    best_row = df_a.loc[df_a["f1"].idxmax()]

    ax.plot(df_a["recall"], df_a["precision"],
            color=color, linewidth=2, marker='o', markersize=4)
    ax.scatter(best_row["recall"], best_row["precision"],
               color='crimson', s=100, zorder=5,
               label=f'Best F1={best_row["f1"]:.3f}\n(thresh={best_row["threshold"]})')
    ax.axhline(base_rate, color='gray', linestyle='--',
               linewidth=1, label=f'Baseline ({base_rate:.1%})')
    ax.set_title(f"{approach}  |  AUC={auc_val:.3f}", fontsize=11)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle("Precision-Recall curves — three relevance score approaches", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "pr_curves.png", dpi=150)
plt.show()

In [ ]:
# F1 vs threshold
fig, ax = plt.subplots(figsize=(12, 5))
for approach, color in zip(APPROACHES, ["steelblue", "darkorange", "green"]):
    df_a = eval_df[eval_df["approach"] == approach]
    ax.plot(df_a["threshold"], df_a["f1"],
            label=approach, color=color, linewidth=2)
ax.set_title("F1 score vs threshold — all three approaches")
ax.set_xlabel("Score threshold")
ax.set_ylabel("F1")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "f1_vs_threshold.png", dpi=150)
plt.show()

## Composite relevance score

Weight each approach by its AUC excess over 0.5 (excess over random).
Approaches that don't beat random receive zero weight.

In [ ]:
auc_by_approach = eval_df.groupby("approach")["auc"].first()
print("AUC by approach:")
print(auc_by_approach)

valid = auc_by_approach[auc_by_approach > 0.5]
if len(valid) == 0:
    print("\nNo approach beats random — using equal weights")
    valid = auc_by_approach

auc_weights = (valid - 0.5) / (valid - 0.5).sum()
print("\nComposite weights (AUC excess over 0.5):")
print(auc_weights.round(3))

panel_scored["score_composite"] = sum(
    panel_scored[approach] * weight
    for approach, weight in auc_weights.items()
    if approach in panel_scored.columns
)

auc_comp = roc_auc_score(y_true, panel_scored["score_composite"].fillna(0).values)
print(f"\nComposite AUC: {auc_comp:.4f}")

## Dynamic country ranking

In [ ]:
ranking_frames = []
for country in countries:
    mask = panel_scored["lhg_country"] == country
    df_c = panel_scored[mask].copy().sort_values("date")
    df_c["score_rolling"] = df_c["score_composite"].rolling(14, min_periods=3).mean()
    ranking_frames.append(df_c)

ranked = pd.concat(ranking_frames).sort_values(["date", "lhg_country"])

# Only rank rows where score_rolling is not NaN
ranked["rank"] = (
    ranked.groupby("date")["score_rolling"]
    .rank(ascending=False, method="min", na_option="keep")  # keeps NaN as NaN
    .astype("Int64")  # nullable integer — handles NaN gracefully
)

latest_date = ranked["date"].max()
top10 = (
    ranked[ranked["date"] == latest_date]
    .dropna(subset=["score_rolling"])          # exclude countries with no rolling score
    .sort_values("rank")
    .head(10)
    [["lhg_country", "score_composite", "score_rolling", "rank"]]
)
print(f"Top 10 destinations on {latest_date}:")
print(top10.to_string(index=False))

In [ ]:
# Rank trajectory for top 8 countries
top8 = top10["lhg_country"].head(8).tolist()

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.get_cmap('tab10', len(top8))

for i, country in enumerate(top8):
    df_c = ranked[ranked["lhg_country"] == country].sort_values("date")
    ax.plot(df_c["date"], df_c["rank"],
            label=country, color=cmap(i), linewidth=1.5, alpha=0.8)

ax.invert_yaxis()
ax.set_title("Relevance rank over time — top 8 countries (rank 1 = most relevant)")
ax.set_xlabel("Date")
ax.set_ylabel("Rank")
ax.legend(loc='upper right', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "rank_trajectory.png", dpi=150)
plt.show()

In [ ]:
# Monthly score heatmap
ranked["month"] = pd.to_datetime(ranked["date"]).dt.to_period("M").astype(str)

monthly_scores = (
    ranked
    .groupby(["lhg_country", "month"])["score_composite"]
    .mean()
    .unstack("month")
)
monthly_scores = monthly_scores.loc[
    monthly_scores.mean(axis=1).sort_values(ascending=False).index
]

fig, ax = plt.subplots(figsize=(14, max(8, len(monthly_scores) * 0.35)))
sns.heatmap(
    monthly_scores,
    cmap="YlOrRd",
    linewidths=0.3,
    ax=ax,
    cbar_kws={"label": "Mean composite relevance score"},
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7}
)
ax.set_title("Monthly average relevance score by country", fontsize=12)
ax.set_yticks([])
ax.set_xlabel("Month")
ax.set_ylabel("Countries")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "score_heatmap_monthly.png", dpi=150)
plt.show()

## Approach comparison summary

In [ ]:
# Score distributions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
score_cols = ["score_A", "score_B", "score_C", "score_composite"]
labels     = ["A: Weighted linear", "B: Sentiment-first",
              "C: Momentum", "Composite"]
colors     = ["steelblue", "darkorange", "green", "purple"]

for ax, col, label, color in zip(axes, score_cols, labels, colors):
    panel_scored[col].hist(bins=50, ax=ax, color=color, edgecolor='none', alpha=0.75)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Score")

plt.suptitle("Score distributions", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "score_distributions.png", dpi=150)
plt.show()

print("Correlation between approaches:")
print(panel_scored[score_cols].corr().round(3))

In [ ]:
# Final summary table
approach_labels = {
    "score_A": "A: Weighted linear",
    "score_B": "B: Sentiment-first",
    "score_C": "C: Momentum",
}
n_days = panel_scored["date"].nunique()
summary_rows = []

for approach, label in approach_labels.items():
    df_a     = eval_df[eval_df["approach"] == approach]
    best_row = df_a.loc[df_a["f1"].idxmax()]
    summary_rows.append({
        "Approach":      label,
        "AUC":           best_row["auc"],
        "Best F1":       best_row["f1"],
        "Precision":     best_row["precision"],
        "Recall":        best_row["recall"],
        "Threshold":     best_row["threshold"],
        "Flags/day":     round(best_row["n_flagged"] / n_days, 1),
    })

summary_df = pd.DataFrame(summary_rows)
print("Approach comparison")
print(summary_df.to_string(index=False))
summary_df.to_csv(OUTPUT_PATH / "approach_comparison.csv", index=False)

Softer target + high-signal country filter:

In [ ]:
# Alternative evaluation: softer target + high-signal countries

HIGH_SIGNAL_COUNTRIES = ["PT", "IT", "ES", "HR", "GR", "FR", "AL", "IE"]

# # Softer target: above-average bookings (not just rare spikes)
# panel_scored["above_average"] = (
#     panel_scored["pax"] > panel_scored["pax_roll28_baseline"]
# ).astype(int)

# Softer target: top 33% of booking days per country (stricter than simple above-average)
panel_scored["above_average"] = (
    panel_scored["pax"] > panel_scored.groupby("lhg_country")["pax"]
    .transform(lambda x: x.quantile(0.67))
).astype(int)

print(f"Above-average base rate (all countries) : {panel_scored['above_average'].mean():.1%}")
print(f"Spike base rate (all countries)         : {panel_scored['pax_spike_positive'].mean():.1%}")

# Build forward label with softer target and 28-day window
panel_scored["future_above_avg"] = build_forward_spike_label(
    panel_scored, "above_average", 28
)

print(f"\nFuture above-avg base rate: {panel_scored['future_above_avg'].mean():.1%}")

Re-evaluate all approaches with new settings:

In [ ]:
# Re-run evaluation: softer target, 28d window, all countries then high-signal

def evaluate_approach(df, score_col, target_col, thresholds):
    y_true = df[target_col].values
    scores = df[score_col].fillna(0).values
    try:
        auc = roc_auc_score(y_true, scores)
    except Exception:
        auc = np.nan

    rows = []
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        if y_pred.sum() == 0:
            continue
        rows.append({
            "threshold": round(thresh, 2),
            "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
            "auc":       round(auc, 4),
            "n_flagged": int(y_pred.sum()),
        })
    return pd.DataFrame(rows)

THRESHOLDS = np.arange(0.1, 0.91, 0.05)
summary_rows = []

for scope_name, df_scope in [
    ("All countries",        panel_scored),
    ("High-signal only",     panel_scored[panel_scored["lhg_country"].isin(HIGH_SIGNAL_COUNTRIES)]),
]:
    for approach in ["score_A", "score_B", "score_C"]:
        df_eval  = evaluate_approach(df_scope, approach, "future_above_avg", THRESHOLDS)
        best_row = df_eval.loc[df_eval["f1"].idxmax()]
        summary_rows.append({
            "Scope":      scope_name,
            "Approach":   approach,
            "AUC":        best_row["auc"],
            "Best F1":    best_row["f1"],
            "Precision":  best_row["precision"],
            "Recall":     best_row["recall"],
            "Threshold":  best_row["threshold"],
        })

summary_alt = pd.DataFrame(summary_rows)
print("Alternative evaluation (softer target, 28d window)")
print(summary_alt.to_string(index=False))
summary_alt.to_csv(OUTPUT_PATH / "approach_comparison_alt.csv", index=False)

Side by side AUC comparison plot:

In [ ]:
# Visual comparison: original vs alternative evaluation
original_aucs = {
    "score_A": 0.4377,
    "score_B": 0.4493,
    "score_C": 0.5315,
}

alt_all    = summary_alt[summary_alt["Scope"] == "All countries"].set_index("Approach")["AUC"]
alt_high   = summary_alt[summary_alt["Scope"] == "High-signal only"].set_index("Approach")["AUC"]

x     = np.arange(3)
width = 0.25
approaches = ["score_A", "score_B", "score_C"]
labels     = ["A: Weighted\nlinear", "B: Sentiment\nfirst", "C: Momentum"]

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width, [original_aucs[a] for a in approaches],
               width, label="Original (spike, 14d)", color="steelblue", alpha=0.8)
bars2 = ax.bar(x,          [alt_all.get(a, 0)    for a in approaches],
               width, label="Softer target, 28d (all countries)", color="darkorange", alpha=0.8)
bars3 = ax.bar(x + width,  [alt_high.get(a, 0)   for a in approaches],
               width, label="Softer target, 28d (high-signal only)", color="green", alpha=0.8)

ax.axhline(0.5, color='crimson', linestyle='--', linewidth=1.5, label="Random (AUC=0.5)")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("AUC")
ax.set_ylim(0.3, 0.85)
ax.set_title("AUC comparison: original vs alternative evaluation settings")
ax.legend(fontsize=9)

# Annotate bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.3f}",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "auc_comparison.png", dpi=150)
plt.show()

print("\nKey takeaway:")
print("  Original evaluation (rare spikes, 14d) - hardest possible target")
print("  Softer target (above-average, 28d)     - more realistic operational use case")
print("  High-signal countries only             - where Reddit signal is actually meaningful")

## Save outputs

In [ ]:
pl.from_pandas(
    panel_scored[["date", "lhg_country", "pax", "pax_zscore",
                  "pax_spike_positive", "future_spike",
                  "score_A", "score_B", "score_C", "score_composite"]]
).write_parquet(OUTPUT_PATH / "panel_scored.parquet")

ranked[["date", "lhg_country", "score_composite",
        "score_rolling", "rank"]].to_parquet(
    OUTPUT_PATH / "country_rankings.parquet", index=False
)

eval_df.to_csv(OUTPUT_PATH / "score_evaluation.csv", index=False)
summary_df.to_csv(OUTPUT_PATH / "approach_comparison.csv", index=False)
summary_alt.to_csv(OUTPUT_PATH / "approach_comparison_alt.csv", index=False)

print("Saved:")
print("  outputs/panel_scored.parquet")
print("  outputs/country_rankings.parquet")
print("  outputs/score_evaluation.csv")
print("  outputs/approach_comparison.csv")
print("  outputs/approach_comparison_alt.csv")